SQL analysis — business queries
Google Colab + SQLite

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded!")

Libraries loaded!


In [3]:
from google.colab import drive
drive.mount('/content/drive')

print("Drive mounted successfully!")

Mounted at /content/drive
Drive mounted successfully!


In [9]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import sqlite3

df = pd.read_csv('/content/drive/MyDrive/olist_project/master_table.csv')

conn = sqlite3.connect('olist.db')
df.to_sql('orders', conn, if_exists='replace', index=False)
print("SQLite DB created with orders table!")
print(f"Rows loaded: {len(df)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SQLite DB created with orders table!
Rows loaded: 110840


Helper function to run SQL queries

In [10]:
def run_sql(query):
    return pd.read_sql_query(query, conn)

print("SQL helper ready! Use run_sql('SELECT ...')")

SQL helper ready! Use run_sql('SELECT ...')


Monthly revenue trend

In [11]:
q1 = run_sql("""
    SELECT
        order_month,
        COUNT(DISTINCT order_id)   AS total_orders,
        ROUND(SUM(payment_value),2) AS total_revenue,
        ROUND(AVG(payment_value),2) AS avg_order_value
    FROM orders
    GROUP BY order_month
    ORDER BY order_month
""")
print(q1)

   order_month  total_orders  total_revenue  avg_order_value
0      2016-09             1            NaN              NaN
1      2016-10           265       62185.82           196.17
2      2016-12             1          19.62            19.62
3      2017-01           750      178282.10           192.95
4      2017-02          1653      327928.86           175.46
5      2017-03          2546      508767.44           174.41
6      2017-04          2303      457050.31           177.36
7      2017-05          3546      707042.90           174.66
8      2017-06          3135      590223.90           167.49
9      2017-07          3872      720446.68           161.54
10     2017-08          4193      850611.08           175.35
11     2017-09          4150     1003326.07           210.30
12     2017-10          4478     1012420.44           192.62
13     2017-11          7289     1559739.87           182.70
14     2017-12          5513     1023434.55           164.70
15     2018-01          

Query 2 — Top 10 product categories by revenue

In [12]:
q2 = run_sql("""
    SELECT
        product_category_name_english AS category,
        COUNT(*)                       AS total_orders,
        ROUND(SUM(payment_value),2)    AS total_revenue,
        ROUND(AVG(review_score),2)     AS avg_rating
    FROM orders
    GROUP BY category
    ORDER BY total_revenue DESC
    LIMIT 10
""")
print(q2)

                category  total_orders  total_revenue  avg_rating
0         bed_bath_table         11107     1723932.14        3.93
1          health_beauty          9519     1625923.50        4.20
2  computers_accessories          7708     1563315.62        3.99
3        furniture_decor          8239     1408110.04        3.96
4          watches_gifts          5869     1388699.25        4.08
5         sports_leisure          8489     1357249.46        4.17
6             housewares          6819     1072820.85        4.11
7                   auto          4158      835782.91        4.13
8           garden_tools          4282      813055.77        4.09
9             cool_stuff          3727      746763.39        4.20


Query 3 — Late delivery rate by state (window function)

In [13]:
q3 = run_sql("""
    SELECT
        customer_state,
        COUNT(*)                             AS total_orders,
        SUM(is_late)                         AS late_orders,
        ROUND(SUM(is_late)*100.0/COUNT(*),1) AS late_pct,
        ROUND(AVG(delivery_delay_days),1)    AS avg_delay_days
    FROM orders
    GROUP BY customer_state
    ORDER BY late_pct DESC
    LIMIT 10
""")
print(q3)

  customer_state  total_orders  late_orders  late_pct  avg_delay_days
0             AL           431           90      20.9            -8.7
1             MA           805          144      17.9           -10.0
2             SE           375           61      16.3           -10.0
3             CE          1431          195      13.6           -11.1
4             PI           524           71      13.5           -11.5
5             BA          3703          442      11.9           -11.0
6             RJ         14224         1653      11.6           -12.0
7             PA          1061          119      11.2           -14.2
8             RR            46            5      10.9           -18.3
9             PB           587           63      10.7           -13.1


Query 4 — Top sellers by revenue with ranking

In [14]:
q4 = run_sql("""
    WITH seller_stats AS (
        SELECT
            seller_id,
            seller_state,
            COUNT(DISTINCT order_id)    AS orders_handled,
            ROUND(SUM(payment_value),2) AS revenue,
            ROUND(AVG(review_score),2)  AS avg_rating
        FROM orders
        GROUP BY seller_id, seller_state
    )
    SELECT
        seller_id,
        seller_state,
        orders_handled,
        revenue,
        avg_rating,
        RANK() OVER (ORDER BY revenue DESC) AS revenue_rank
    FROM seller_stats
    LIMIT 15
""")
print(q4)


                           seller_id seller_state  orders_handled    revenue  \
0   7c67e1448b00f6e969d365cea6b010ab           SP             973  510915.44   
1   1025f0e2d44d7041d6cf58b6550e0bfa           SP             910  310186.70   
2   4a3ca9315b744ce9f8e9374361493884           SP            1772  300724.29   
3   1f50f920176fa81dab994f9023523100           SP            1399  291526.94   
4   53243585a1d6dc2643021fd1853d8905           BA             348  279843.42   
5   da8622b14eb17ae2831f4ac5b9dab84a           SP            1311  276093.09   
6   4869f7a5dfa277a7dca6462dcf3b52b2           SP            1124  261532.48   
7   955fee9216a65b617aa5c0531780ce60           SP            1261  232228.24   
8   fa1c13f2614d7b5c4749cbc52fecda94           SP             578  203262.00   
9   6560211a19b47992c3666cc44a7e94c0           SP            1819  176467.46   
10  7e93a43ef30c4f03f38b393420bc753a           SP             319  174053.79   
11  7a67c85e85bb2ce8582c35f2203ad736    

Query 5 — Review score vs delivery delay

In [15]:
q5 = run_sql("""
    SELECT
        review_score,
        COUNT(*)                          AS orders,
        ROUND(AVG(delivery_delay_days),1) AS avg_delay,
        ROUND(AVG(payment_value),2)       AS avg_spend
    FROM orders
    GROUP BY review_score
    ORDER BY review_score
""")
print(q5)

   review_score  orders  avg_delay  avg_spend
0           1.0   12575       -6.0     245.60
1           2.0    3700       -9.7     196.91
2           3.0    9242      -11.1     175.15
3           4.0   21184      -12.5     170.83
4           5.0   64139      -13.3     168.82


Save all query results to Drive


In [16]:
PATH = '/content/drive/MyDrive/olist_project/'
q1.to_csv(PATH + 'sql_monthly_revenue.csv', index=False)
q2.to_csv(PATH + 'sql_top_categories.csv', index=False)
q3.to_csv(PATH + 'sql_late_by_state.csv', index=False)
q4.to_csv(PATH + 'sql_top_sellers.csv', index=False)
q5.to_csv(PATH + 'sql_review_vs_delay.csv', index=False)
print("All query results saved!")

All query results saved!
